In [1]:
import numpy as np
import pandas as pd
import argparse
from ast import literal_eval

In [11]:
def zero_shot(df, refLogits, mutLogits, shift):
    res = []
    for i, ((_, row), ref, mut) in enumerate(zip(df.iterrows(), refLogits, mutLogits)):
        nucleotides = "ACGT"
        left_pos = literal_eval(row['Left5_Positions']) if isinstance(row['Left5_Positions'], str) else row['Left5_Positions']
        right_pos = literal_eval(row['Right5_Positions']) if isinstance(row['Right5_Positions'], str) else row['Right5_Positions']
        left_pos = [pos - 1 - shift for pos in left_pos]
        right_pos = [pos - 1  + shift for pos in right_pos]
        ref = ref[left_pos + right_pos]
        
        left_pos = list(range(4092, 4097))
        right_pos = list(range(4097, 4102))
        left_pos = [pos - 1 - shift for pos in left_pos]
        right_pos = [pos - 1  + shift for pos in right_pos]
        mut = mut[left_pos + right_pos]
        
        upstream_idx = 4091 - shift
        downstream_idx = 4096 + shift
        curSeq = row['MutSeq'][upstream_idx:(upstream_idx + 5)] + row['MutSeq'][downstream_idx:(downstream_idx + 5)]
        scores = []
        for idx, nt in enumerate(curSeq):
            if nt in nucleotides:
                refProb = ref[idx, nucleotides.index(nt)]
                mutProb = mut[idx, nucleotides.index(nt)]
                scores.append(np.log(mutProb / refProb))
            else:
                scores.append(0)
        res.append(scores)
    return res

def zero_shot_ref_seq(df, refLogits):
    res = []
    for i, ((_, row), ref) in enumerate(zip(df.iterrows(), refLogits)):
        nucleotides = "ACGT"
        left_pos = literal_eval(row['Left5_Positions']) if isinstance(row['Left5_Positions'], str) else row['Left5_Positions']
        right_pos = literal_eval(row['Right5_Positions']) if isinstance(row['Right5_Positions'], str) else row['Right5_Positions']
        left_pos = left_pos[-1] + 1 - 1
        right_pos = right_pos[0] - 1    
        ref = ref[range(left_pos, right_pos)]
        
        curSeq = row['RefSeq'][left_pos:right_pos]
        scores = []
        for idx, nt in enumerate(curSeq):
            if nt in nucleotides:
                refProb = ref[idx, nucleotides.index(nt)]
                mutProb = np.delete(ref[idx,:], nucleotides.index(nt)).max()
                scores.append(np.log(mutProb / refProb))
            else:
                scores.append(0)
        res.append(scores)
    return res

In [3]:
loaded = np.load('../../results/SV_effect/outputs/Ath_Simulated_DEL_Len_1-50_RefSeq_nomask_pcv2-l48-d1536.npz')
refLogits = loaded['logits']

loaded = np.load('../../results/SV_effect/outputs/Ath_Simulated_DEL_Len_1-50_MutSeq_nomask_pcv2-l48-d1536.npz')
mutLogits = loaded['logits']

In [4]:
refLogits.shape, mutLogits.shape

((39976, 8192, 4), (39976, 8192, 4))

In [5]:
for shift in [0, 5, 10, 15, 20]:
    df = pd.read_csv('../../results/SV_effect/inputs/Ath_Simulated_DEL_Len_1-50.tsv', sep='\t')
    df['Location'] = df['Chromosome'].astype(str) + ":" + df['Start'].astype(str) + "-" + df['End'].astype(str)
    res = zero_shot(df, refLogits, mutLogits, shift=shift)
    score_df = pd.DataFrame(res, columns=[f'score_{i}' for i in range(10)])
    df = pd.concat([df, score_df], axis=1)
    df.to_csv(f'../../results/SV_effect/outputs/Ath_Simulated_DEL_Len_1-50_nomask_shift_{shift}_score.tsv', sep='\t', index=False)

In [22]:
for shift in [0, 5, 10, 15, 20]:
    df = pd.read_csv('../../results/SV_effect/inputs/Ath_Simulated_DEL_Len_1-50.tsv', sep='\t')
    df['Location'] = df['Chromosome'].astype(str) + ":" + df['Start'].astype(str) + "-" + df['End'].astype(str)
    res = zero_shot_ref_seq(df, refLogits)
    score_df = [np.mean(i) for i in res]
    df['mean_pcv2_del'] = score_df
    df.to_csv(f'../../results/SV_effect/outputs/Ath_Simulated_DEL_Len_1-50_ref_seq_score.tsv', sep='\t', index=False)